# 20M Baseline — Google Colab
Run in order. Phase 7 supports setup through two-step smoke training; checkpointing, full training, evaluation, and plots arrive in later phases.

## 00 - Environment

In [ ]:
import os, platform, subprocess, sys, torch
print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

## 01 - GPU verification

In [ ]:
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GiB:', round(props.total_memory / 2**30, 2))
    print('BF16 supported:', torch.cuda.is_bf16_supported())
    subprocess.run(['nvidia-smi'])
    vram_gib = props.total_memory / 2**30
    recommendation = 8 if vram_gib >= 16 else 4 if vram_gib >= 12 else 2 if vram_gib >= 8 else 1
    print('Suggested initial micro-batch:', recommendation)
else:
    print('No GPU. Use Runtime > Change runtime type > GPU, or continue with CPU smoke checks.')

## 02 - Repository setup

In [ ]:
REPOSITORY_URL = 'https://github.com/<YOUR-USER>/20m-llm-lab.git'  # replace
REPOSITORY_DIR = '/content/20m-llm-lab'
if not os.path.exists(REPOSITORY_DIR):
    subprocess.run(['git', 'clone', REPOSITORY_URL, REPOSITORY_DIR], check=True)
os.chdir(REPOSITORY_DIR)
print('Working directory:', os.getcwd())

## 03 - Dependency installation

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'PyYAML==6.0.3', 'tokenizers==0.23.2', 'datasets==5.0.1'], check=True)
import datasets, tokenizers
print('datasets:', datasets.__version__, 'tokenizers:', tokenizers.__version__)

## 04 - Dataset (explicit smoke export)

In [ ]:
subprocess.run([sys.executable, '-m', 'training.dataset', 'export', '--mode', 'smoke', '--output-dir', 'data/raw/tinystories_smoke'], check=True)

## 05 - Tokenizer

In [ ]:
subprocess.run([sys.executable, '-m', 'tokenizer.train_tokenizer', '--input', 'data/raw/tinystories_smoke/train.txt', '--output-dir', 'artifacts/tokenizers/tinystories_smoke_bpe_32k'], check=True)
subprocess.run([sys.executable, '-m', 'training.dataset', 'pack', '--input', 'data/raw/tinystories_smoke/train.txt', '--tokenizer', 'artifacts/tokenizers/tinystories_smoke_bpe_32k/tokenizer.json', '--output', 'artifacts/datasets/smoke_train.bin'], check=True)

## 06 - Model configuration and 07 - Parameter count

In [ ]:
from model import BaselineTransformer, calculate_parameter_breakdown, load_baseline_config
config = load_baseline_config('configs/baseline_20m.yaml')
model = BaselineTransformer(config.model)
print('Model parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
print('Deterministic count:', f'{calculate_parameter_breakdown(config.model).total:,}')

## 07 - CPU/basic tests
## 08 - GPU qualification run
Run 100 optimizer steps only after inspecting the resolved GPU/precision output.
## 09 - Inspect qualification metrics

In [ ]:
import torch
tokens = torch.randint(0, config.model.vocab_size, (1, 8))
print('Logits:', tuple(model(tokens).shape))
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
print('Qualification: micro batch=4, accumulation=8, effective batch=32, tokens/step=16384')
subprocess.run([sys.executable, '-m', 'training.train', '--dataset', 'artifacts/datasets/smoke_train.bin', '--steps', '100', '--micro-batch', '4', '--accumulation', '8', '--device', 'auto', '--precision', 'auto', '--run-name', 'gpu_qualification'], check=True)

## 10 - Real training
**Deferred to Phase 10.**

## 11 - Validation
**Deferred to Phase 9.**

## 12 - Checkpoint/resume
**Deferred to Phase 8.**

## 13 - Text generation
**Deferred to Phase 9.**

## 14 - Loss plots
**Deferred to Phase 9.**

## 15 - Experiment summary
**Deferred to Phase 10.**

## 16 - Save artifacts
Prepare persistent Drive storage now; Phase 8 will write resumable checkpoints there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/20m-llm-lab'
os.makedirs(DRIVE_PROJECT_DIR, exist_ok=True)
print('Persistent project directory:', DRIVE_PROJECT_DIR)